In [35]:
# =================================================================================================
# Project: NHANES Harmonized Dataset Verification
# Dataset: Demographics
# Years: 2003–2014
# Purpose: Verify Kaggle harmonized demographics data against raw NHANES DEMO files.
# Source Raw Data: CDC NHANES DEMO_C through DEMO_H XPT files
# Harmonized Data: Kaggle NHANES harmonized dataset
# Developer: Neha Khairnar
# https://www.kaggle.com/datasets/nguyenvy/nhanes-19882018
# =================================================================================================
# Step 1: Imports
# Step 2: Define file paths
# Step 3: Load raw NHANES DEMO files
# Step 4: Load Patel harmonized demographics file
# Step 5: Compare row counts by cycle
# Step 6: Compare SEQN values
# Step 7: Compare key demographic variables
# Step 8: Generate verification report
# --------------------------------------------------
import pandas as pd
from pathlib import Path

# =================================================================================================
# Step 1: Define Paths
# =================================================================================================

BASE_DIR = Path().resolve().parent
DATA_DIR = BASE_DIR / "data" / "datasets"/ "NHANE_Continuous_DemographicsData_Appended_2003_2014_Data_only"


OUTPUT_DIR = BASE_DIR / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

KAGGLE_DIR = BASE_DIR / "data" / "datasets"/ "NHANE_HARMONIZED" /"archive"
kaggle_file = KAGGLE_DIR / "demographics_clean.csv"

# =================================================================================================
# Step 2: Raw NHANES DEMO Files: 2003–2014
# =================================================================================================


# List all demographic files (update names as per your files)
demo_files = {
    "2003-2004": "DEMO_C.XPT",
    "2005-2006": "DEMO_D.XPT",
    "2007-2008": "DEMO_E.XPT",
    "2009-2010": "DEMO_F.XPT",
    "2011-2012": "DEMO_G.XPT",
    "2013-2014": "DEMO_H.XPT",
}

# =================================================================================================
# Step 3: Load Raw NHANES Demographics
# =================================================================================================

raw_list = []

for filename in demo_files.values():
    df = pd.read_sas(DATA_DIR / filename, format="xport")
    raw_list.append(df)

raw_demo = pd.concat(raw_list, ignore_index=True)

raw_demo.shape
# =================================================================================================
# Step 4: Load Patel Kaggle Harmonized Demographics
# =================================================================================================
"""
| SDDSRVYR | Cycle            |
| -------- | ---------------- |
| 1        | NHANES III       |
| 2        | NHANES 1999-2002 |
| 3        | NHANES 2003-2004 |
| 4        | NHANES 2005-2006 |
| 5        | NHANES 2007-2008 |
| 6        | NHANES 2009-2010 |
| 7        | NHANES 2011-2012 |
| 8        | NHANES 2013-2014 |
| 9        | NHANES 2015-2016 |
| 10       | NHANES 2017-2018 |
"""

    
kaggle_demo = pd.read_csv(kaggle_file)
Kaggle_demo_2003_2014 = kaggle_demo[
    kaggle_demo["SDDSRVYR"].isin([3,4,5,6,7,8])
]

kaggle_demo.shape

# =================================================================================================
# Step 5: Inspect Columns
# =================================================================================================
print("\nInspecting columns for -  raw NHANE vs harmonized kaggle NHANE :")
print("RAW NHANE columns:")
print(raw_demo.columns.tolist())
print("\nKAGGLE NHANE columns:")
print(kaggle_demo.columns.tolist())
print("-----------------------------------------------------")

print("\nInspecting Rows count for - raw NHANE vs harmonized kaggle NHANE :")
print("\nRaw NHANE rows:", len(raw_demo))
print("Kaggle NHANE total rows:", len(kaggle_demo))
print("Kaggle NHANE rows for years 2003-2014:", len(Kaggle_demo_2003_2014))
print("-----------------------------------------------------")

print("\nInspecting Missing for - raw NHANE vs harmonized kaggle NHANE :")
raw_seqn = set(raw_demo["SEQN"])
kaggle_seqn = set(Kaggle_demo_2003_2014["SEQN"])
print("Missing in Kaggle NHANE:", len(raw_seqn - kaggle_seqn))
print("Extra in Kaggle NHANE:", len(kaggle_seqn - raw_seqn))
print("-----------------------------------------------------")
# =================================================================================================
# Step 5: Validate both exist participant IDS SEQN
# =================================================================================================

# Only participants that exist in BOTH datasets
print("\nvalidation check on participants that exist in BOTH - raw NHANE vs harmonized kaggle NHANE :")
verified = raw_demo.merge(
    Kaggle_demo_2003_2014,
    on="SEQN",
    how="inner",
    suffixes=("_RAW", "_KAGGLE")
)

print("Verified participants:", len(verified))
print("-----------------------------------------------------")

# =================================================================================================
# Step 6: Compare every column
# =================================================================================================

##Steps to troubleshoot missing columns . 
## eg.MISSING IN Kaggle below columns found missing in harmonized dataset. To troublehsoot, we need to find below variables.
#There are two saperate CSV in kaggle dataset dictionary_nhanes.csv, which help to troubleshoot nelow missing columns gets harmonized into another variable.
# eg. DMDBORN -> DMDBORN4 because some csvs are using DMDBORN and some cycles are revised to DMDBORN4 or DMDBORN2 so on.
"""
AIALANG
DMDBORN
DMDBORN2
DMDHRBR2
DMDHRBRN
DMQMILIT
INDFMINC
INDHHINC
RIDAGEEX
"""

raw_demo.columns = raw_demo.columns.str.upper()
Kaggle_demo_2003_2014.columns = Kaggle_demo_2003_2014.columns.str.upper()

raw_cols = set(raw_demo.columns)
kaggle_cols = set(Kaggle_demo_2003_2014.columns)

common_cols = sorted(raw_cols.intersection(kaggle_cols))
missing_in_kaggle = sorted(raw_cols - kaggle_cols)
extra_in_kaggle = sorted(kaggle_cols - raw_cols)

print("=" * 80)
print("RAW NHANES COLUMNS:", len(raw_cols))

print("=" * 80)
print("PATEL COLUMNS:", len(kaggle_cols))

print("=" * 80)
print("COMMON COLUMNS:", len(common_cols))

print("=" * 80)
print("MISSING IN Kaggle:", len(missing_in_kaggle))

print("=" * 80)
print("EXTRA IN Kaggle:", len(extra_in_kaggle))


print("=" * 80)
print("\nCOMMON COLUMNS")
for c in common_cols:
    print(c)
print("=" * 80)

print("\nMISSING IN Kaggle")
for c in missing_in_kaggle:
    print(c)
print("=" * 80)

print("\nEXTRA IN Kaggle")
for c in extra_in_kaggle:
    print(c)

print("=" * 80)
# =================================================================================================
# Step 6: Report of which column is missing and need to investigate why
# =================================================================================================
report = pd.DataFrame({
    "column_name": sorted(raw_cols.union(patel_cols))
})

report["exists_in_raw"] = report["column_name"].isin(raw_cols)
report["exists_in_kaggle"] = report["column_name"].isin(kaggle_cols)

report["status"] = report.apply(
    lambda x: "YES"
    if x["exists_in_raw"] and x["exists_in_kaggle"]
    else "NO",
    axis=1
)

report
print("\n-----------------------------------------------------")

# =================================================================================================
# 1. Final Kaggle dataset with ONLY common columns from raw
# =================================================================================================

raw_demo.columns = raw_demo.columns.str.upper()
Kaggle_demo_2003_2014.columns = Kaggle_demo_2003_2014.columns.str.upper()

raw_cols = set(raw_demo.columns)
kaggle_cols = set(Kaggle_demo_2003_2014.columns)

common_cols = sorted(raw_cols.intersection(patel_cols))

kaggle_common_only = Kaggle_demo_2003_2014[common_cols].copy()

print("Kaggle dataset with common raw columns only:")
print(kaggle_common_only.shape)

display(kaggle_common_only.head())

print("\n-----------------------------------------------------")
# =================================================================================================
# 2. Kaggle dataset 
# =================================================================================================
print("Kaggle dataset with all columns :")
print(Kaggle_demo_2003_2014.shape)

display(Kaggle_demo_2003_2014.head())


Inspecting columns for -  raw NHANE vs harmonized kaggle NHANE :
RAW NHANE columns:
['SEQN', 'SDDSRVYR', 'RIDSTATR', 'RIDEXMON', 'RIAGENDR', 'RIDAGEYR', 'RIDAGEMN', 'RIDAGEEX', 'RIDRETH1', 'RIDRETH2', 'DMQMILIT', 'DMDBORN', 'DMDCITZN', 'DMDYRSUS', 'DMDEDUC3', 'DMDEDUC2', 'DMDEDUC', 'DMDSCHOL', 'DMDMARTL', 'DMDHHSIZ', 'INDHHINC', 'INDFMINC', 'INDFMPIR', 'RIDEXPRG', 'DMDHRGND', 'DMDHRAGE', 'DMDHRBRN', 'DMDHREDU', 'DMDHRMAR', 'DMDHSEDU', 'SIALANG', 'SIAPROXY', 'SIAINTRP', 'FIALANG', 'FIAPROXY', 'FIAINTRP', 'MIALANG', 'MIAPROXY', 'MIAINTRP', 'AIALANG', 'WTINT2YR', 'WTMEC2YR', 'SDMVPSU', 'SDMVSTRA', 'DMDFMSIZ', 'DMDBORN2', 'INDHHIN2', 'INDFMIN2', 'DMDHRBR2', 'RIDRETH3', 'RIDEXAGY', 'RIDEXAGM', 'DMQMILIZ', 'DMQADFC', 'DMDBORN4', 'AIALANGA', 'DMDHHSZA', 'DMDHHSZB', 'DMDHHSZE', 'DMDHRBR4']

KAGGLE NHANE columns:
['SEQN', 'SEQN_new', 'AIALANGA', 'DMAETHN', 'DMAETHNR', 'DMARACE', 'DMARACER', 'DMDBORN4', 'DMDCITZN', 'DMDEDUC', 'DMDEDUC2', 'DMDEDUC3', 'DMDFMSIZ', 'DMDHHSIZ', 'DMDHHSZA', 'DMDHHSZB

,AIALANGA,DMDBORN4,DMDCITZN,DMDEDUC,DMDEDUC2,DMDEDUC3,DMDFMSIZ,DMDHHSIZ,DMDHHSZA,DMDHHSZB,...,RIDSTATR,SDDSRVYR,SDMVPSU,SDMVSTRA,SEQN,SIAINTRP,SIALANG,SIAPROXY,WTINT2YR,WTMEC2YR
54998,1.0,1.0,1.0,1.0,NaN,11.0,NaN,2,NaN,NaN,...,2,3,2,39,21005,2.0,1.0,2.0,5512.320949,5824.782465
54999,1.0,1.0,1.0,1.0,NaN,11.0,NaN,6,NaN,NaN,...,2,3,1,41,21006,2.0,1.0,2.0,5422.140453,5564.039715
55000,1.0,1.0,1.0,1.0,NaN,8.0,NaN,5,NaN,NaN,...,2,3,2,35,21007,2.0,1.0,1.0,39764.177412,40591.066325
55001,1.0,1.0,1.0,1.0,NaN,10.0,NaN,7,NaN,NaN,...,2,3,1,32,21008,2.0,1.0,2.0,5599.499351,5696.750596
55002,1.0,1.0,1.0,2.0,3.0,NaN,NaN,2,NaN,NaN,...,2,3,2,31,21009,2.0,1.0,2.0,97593.678977,97731.727244



-----------------------------------------------------
Kaggle dataset with all columns :
(61087, 281)


,SEQN,SEQN_NEW,AIALANGA,DMAETHN,DMAETHNR,DMARACE,DMARACER,DMDBORN4,DMDCITZN,DMDEDUC,...,WTMREP43,WTMREP44,WTMREP45,WTMREP46,WTMREP47,WTMREP48,WTMREP49,WTMREP50,WTMREP51,WTMREP52
54998,21005,C-21005,1.0,NaN,NaN,NaN,NaN,1.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
54999,21006,C-21006,1.0,NaN,NaN,NaN,NaN,1.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
55000,21007,C-21007,1.0,NaN,NaN,NaN,NaN,1.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
55001,21008,C-21008,1.0,NaN,NaN,NaN,NaN,1.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
55002,21009,C-21009,1.0,NaN,NaN,NaN,NaN,1.0,1.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
